# RL fine-tuning &mdash; result graphs

TensorBoard scalar curves for the PPO decoder fine-tuning runs, styled to match
the thesis (Computer Modern serif, colour-blind-safe palette).

**How it scales**

- Register a run once in the `RUNS` dict (short key &rarr; log-folder name under
  `outputs/rl_logs/`); afterwards refer to it everywhere by its key.
- Every figure goes through a single helper, `plot_runs([...])`. Pass one spec to
  draw a single run, or several specs to **overlay** runs on the same axes.
- Each curve is drawn three ways: the faint **raw** signal (the true noise), a
  translucent **band** showing the local spread (an EMA-smoothed deviation band,
  so it stays continuous even for seldom-reported signals), and a bold
  **EMA-smoothed** trend on top.
- Figures are written as vector **PDF** (for `\includegraphics`) and **PNG** into
  `rl_finetuning/notebooks/figures/`.

In [ ]:
import functools
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
from tensorboard.backend.event_processing.event_accumulator import EventAccumulator

# --- Fonts: match the thesis (packages.sty uses the LaTeX default, Computer Modern) ---
# No TeX/dvipng is installed, so we reproduce Computer Modern with matplotlib's
# bundled "cmr10" for text and the "cm" set for maths. Set USE_TEX = True if you
# have a TeX distribution + dvipng and want matplotlib to typeset via LaTeX.
USE_TEX = False
SERIF = ["cmr10", "CMU Serif", "Latin Modern Roman", "STIXGeneral", "DejaVu Serif"]

# --- Colours: Paul Tol "bright" (colour-blind safe, prints well in greyscale) ---
PALETTE = {"blue": "#4477AA", "red": "#EE6677", "green": "#228833", "yellow": "#CCBB44",
           "cyan": "#66CCEE", "purple": "#AA3377", "grey": "#BBBBBB"}
CYCLE = [PALETTE[k] for k in ("blue", "red", "green", "purple", "cyan", "yellow")]

BASE = 11  # base font size in pt (close to a 10-11pt thesis body)
mpl.rcParams.update({
    "text.usetex": USE_TEX,
    "font.family": "serif", "font.serif": SERIF,
    "mathtext.fontset": "cm", "axes.unicode_minus": False,
    "axes.formatter.use_mathtext": True,
    "font.size": BASE, "axes.titlesize": BASE + 1, "axes.labelsize": BASE,
    "xtick.labelsize": BASE - 1, "ytick.labelsize": BASE - 1, "legend.fontsize": BASE - 1,
    "axes.linewidth": 0.8, "axes.edgecolor": "#444444",
    "axes.spines.top": False, "axes.spines.right": False, "axes.axisbelow": True,
    "axes.grid": True, "grid.color": "#CCCCCC", "grid.linewidth": 0.6, "grid.alpha": 0.7,
    "xtick.direction": "out", "ytick.direction": "out",
    "xtick.major.size": 3, "ytick.major.size": 3,
    "lines.linewidth": 1.8, "legend.frameon": False, "legend.handlelength": 1.6,
    "figure.figsize": (5.9, 3.6), "figure.dpi": 120,
    "savefig.dpi": 300, "savefig.bbox": "tight",
    "figure.constrained_layout.use": True,
    "axes.prop_cycle": mpl.cycler(color=CYCLE),
})
print("matplotlib", mpl.__version__,
      "| text serif ->", Path(mpl.font_manager.findfont("serif")).name)

### Fonts &amp; colours

`packages.sty` uses the default LaTeX font, **Computer Modern**. We reproduce it
with matplotlib's bundled **`cmr10`** (Computer Modern Roman) for text and the
**`cm`** set for maths, so the figures match the thesis without needing a TeX
install. Set `USE_TEX = True` if you have a TeX distribution + `dvipng` and want
matplotlib to typeset through LaTeX directly (then the body font becomes whatever
LaTeX uses, i.e. Computer Modern as well).

The palette is **Paul Tol's _bright_** scheme (colour-blind safe, legible in
greyscale): reference `PALETTE["blue"]`, `PALETTE["red"]`, &hellip; explicitly,
or let `plot_runs` pull colours from the cycle automatically.


### Data loading

`run_dir` maps a registry key (or a raw folder name) to its log directory;
`load_scalar` merges **every** `events.*` file in that directory &mdash; these
runs were resumed several times &mdash; and returns aligned `(steps, values)`.


In [ ]:
def find_repo_root(marker="outputs/rl_logs"):
    here = Path.cwd().resolve()
    for d in (here, *here.parents):
        if (d / marker).exists():
            return d
    return here

REPO_ROOT = find_repo_root()
LOG_ROOT = REPO_ROOT / "outputs" / "rl_logs"
FIG_DIR = REPO_ROOT / "rl_finetuning" / "notebooks" / "figures"

# Register a run once here (key -> log-folder name under outputs/rl_logs/);
# afterwards refer to it everywhere by its short key.
RUNS = {
    "fixcritic": "TFV6_PPO_LOCAL_only_NSLTEF_fixcritic",
    "warmstart": "TFV6_PPO_LOCAL_only_NSLTEF_warmstart_routedevpenal_terminalwarmupn5",
}

def run_dir(run):
    """Accept a registry key or a raw folder name."""
    return str(LOG_ROOT / RUNS.get(run, run))

print("repo root:", REPO_ROOT)
print("runs:", list(RUNS))

In [ ]:
@functools.lru_cache(maxsize=None)
def _accumulator(path):
    ea = EventAccumulator(path, size_guidance={"scalars": 0})
    ea.Reload()  # merges every events.* file in the dir (runs were resumed)
    return ea

def list_tags(run):
    """List all scalar tags logged for a run."""
    return sorted(_accumulator(run_dir(run)).Tags().get("scalars", []))

def load_scalar(run, tag):
    """Return (steps, values) arrays for a scalar tag."""
    s = _accumulator(run_dir(run)).Scalars(tag)
    return (np.array([p.step for p in s], float),
            np.array([p.value for p in s], float))

def ema(values, weight):
    """TensorBoard-style bias-corrected exponential moving average."""
    out = np.empty_like(values, float)
    last = 0.0
    debias = 0.0
    for i, v in enumerate(values):
        last = last * weight + (1 - weight) * v
        debias = debias * weight + (1 - weight)
        out[i] = last / debias
    return out

def rolling_band(values, kind="percentile", window=41, q=(25, 75), k=1.0):
    """Lower/upper envelope from a centred rolling window (used by non-default
    band kinds). Can look blocky on sparse / quantized signals; the edges are
    EMA-smoothed in `add_trace` to counter that."""
    s = pd.Series(values, dtype=float)
    roll = s.rolling(window, center=True, min_periods=1)
    if kind == "percentile":
        return roll.quantile(q[0] / 100).to_numpy(), roll.quantile(q[1] / 100).to_numpy()
    if kind == "std":
        m = roll.mean().to_numpy()
        sd = roll.std().fillna(0).to_numpy()
        return m - k * sd, m + k * sd
    if kind == "minmax":
        return roll.min().to_numpy(), roll.max().to_numpy()
    raise ValueError(f"unknown band kind: {kind!r}")

def band_bounds(vals, center, *, kind="deviation", band_k=1.0,
                band_smooth=0.9, smoothing=0.9, band_kw=None):
    """Lower/upper band describing the local noise.

    Default ``kind="deviation"`` builds the band as ``center +/- band_k * s``,
    where ``s`` is an EMA-smoothed local standard deviation of the raw signal
    about the smoothed line. Both the centre and the width are EMA outputs, so
    the band is continuous and never blocky -- ideal for signals that are
    reported seldom or take few distinct values (e.g. success / route-finish
    rates). The window-based kinds (``"percentile"``, ``"std"``, ``"minmax"``)
    remain available and get their edges EMA-smoothed to reduce step artefacts.
    """
    if kind == "deviation":
        spread = np.sqrt(ema((vals - center) ** 2, smoothing))
        return center - band_k * spread, center + band_k * spread
    lo, hi = rolling_band(vals, kind=kind, **(band_kw or {}))
    if band_smooth:
        lo, hi = ema(lo, band_smooth), ema(hi, band_smooth)
    return lo, hi

def draw_series(ax, steps, raw, center, label, color, *, xdiv=1e6,
                show_raw=True, raw_alpha=0.16, band="deviation", band_alpha=0.18,
                band_k=1.0, band_smooth=0.9, band_clip=None, band_kw=None,
                smoothing=0.9, smooth_from=None):
    """Draw the standard 3-layer curve from pre-computed series: faint raw signal
    + smooth spread band + bold smoothed centre line. Both `add_trace` (a single
    tag) and `plot_effective_collision` (a derived ratio) build on this so the
    look stays identical everywhere.

    `band_clip=(lo, hi)` clamps the band to a valid range (e.g. `(0, 1)` for a
    rate). `smooth_from` (raw step units, e.g. 10_000) hides the bold smoothed
    line before that step, where the EMA is still warming up.
    """
    x = steps / xdiv
    if show_raw and raw is not None:
        ax.plot(x, raw, color=color, lw=0.7, alpha=raw_alpha, zorder=1)
    if band is not None and raw is not None:
        lo, hi = band_bounds(raw, center, kind=band, band_k=band_k,
                             band_smooth=band_smooth, smoothing=smoothing,
                             band_kw=band_kw)
        if band_clip is not None:
            lo, hi = np.clip(lo, *band_clip), np.clip(hi, *band_clip)
        ax.fill_between(x, lo, hi, color=color, alpha=band_alpha, lw=0, zorder=2)
    sm = np.array(center, float)
    if smooth_from is not None:
        sm = np.where(steps >= smooth_from, sm, np.nan)  # drop EMA warm-up
    ax.plot(x, sm, color=color, lw=1.9, label=label, zorder=3)

def add_trace(ax, run, tag, label, color, *, smoothing=0.9, **kw):
    """Draw one curve for a single scalar tag (raw + band + EMA-smoothed line)."""
    steps, vals = load_scalar(run, tag)
    draw_series(ax, steps, vals, ema(vals, smoothing), label, color,
                smoothing=smoothing, **kw)

def save_fig(fig, name, formats=("pdf", "png")):
    FIG_DIR.mkdir(parents=True, exist_ok=True)
    for fmt in formats:
        fig.savefig(FIG_DIR / f"{name}.{fmt}")
    print("saved:", ", ".join(f"{name}.{f}" for f in formats), "->", FIG_DIR)

def plot_runs(specs, *, tag=None, ylabel="",
              xlabel=r"Environment steps ($\times 10^{6}$)",
              title=None, xlim=None, ylim=None, hline=None, legend_loc="lower right",
              figsize=None, save=None, **trace_kw):
    """Plot one or several runs on shared axes. Pass several specs to overlay."""
    fig, ax = plt.subplots(figsize=figsize)
    if hline is not None:
        ax.axhline(hline, color="#888888", lw=0.8, ls=(0, (4, 3)), zorder=0)
    for i, sp in enumerate(specs):
        add_trace(ax, sp["run"], sp.get("tag", tag), sp["label"],
                  sp.get("color", CYCLE[i % len(CYCLE)]), **trace_kw)
    ax.set_xlabel(xlabel)
    ax.set_ylabel(ylabel)
    if title:
        ax.set_title(title)
    if ylim is not None:
        ax.set_ylim(*ylim)
    if xlim is not None:
        ax.set_xlim(*xlim)
    ax.margins(x=0.01)
    ax.legend(loc=legend_loc)
    if save:
        save_fig(fig, save)
    return fig, ax

# --- Effective collision rate -------------------------------------------------
# All infractions/* series are mutually-exclusive terminal-outcome fractions
# that sum to 1, so a subset defines a conditional probability. We only keep the
# episodes where a junction-collision outcome could actually be *observed*: the
# ego either crashed (`collision`) or drove all the way through
# (`finished_route`). ran_red_light / ran_stop_sign are terminal *at the stop
# line* -- the episode ends the instant the ego crosses it, before it traverses
# the conflict zone -- so those episodes are censored at entry and can never
# register a junction collision; padding the denominator with them would
# re-introduce the very deflation we set out to remove. timeout / ego_blocked /
# route_deviation never reach the junction at all. Hence: collision + finished.
ENTERED_OUTCOMES = ("collision", "finished_route")

def effective_collision(run, *, entered=ENTERED_OUTCOMES, smoothing=0.9):
    """Collision rate conditioned on completing the intersection one way or
    another:

        eff = collision / (sum of `entered` outcomes)
            = collision / (collision + finished_route)

    Returns (steps, raw, center). `raw` is the per-update ratio (noisy; used for
    the faint trace + band); `center` is the ratio of the EMA-smoothed numerator
    and denominator -- a pooled estimate that stays clean even when individual
    updates contain few episodes (much smoother than EMA-ing the raw ratio).
    """
    steps, coll = load_scalar(run, "infractions/collision")
    den = np.zeros_like(coll)
    for k in entered:
        _, v = load_scalar(run, f"infractions/{k}")
        den += v
    raw = coll / np.clip(den, 1e-6, None)
    center = ema(coll, smoothing) / np.clip(ema(den, smoothing), 1e-6, None)
    return steps, raw, center

def plot_effective_collision(specs, *, entered=ENTERED_OUTCOMES,
                             ylabel="Effective collision rate (0-1)",
                             xlabel=r"Environment steps ($\times 10^{6}$)",
                             title=None, xlim=None, ylim=(-0.02, 1.0),
                             legend_loc="upper right", smoothing=0.9,
                             figsize=None, save=None, **trace_kw):
    """Plot the entered-conditioned collision rate for one or several runs."""
    fig, ax = plt.subplots(figsize=figsize)
    for i, sp in enumerate(specs):
        steps, raw, center = effective_collision(
            sp["run"], entered=entered, smoothing=smoothing)
        draw_series(ax, steps, raw, center, sp["label"],
                    sp.get("color", CYCLE[i % len(CYCLE)]),
                    smoothing=smoothing, band_clip=(0, 1), **trace_kw)
    ax.set_xlabel(xlabel)
    ax.set_ylabel(ylabel)
    if title:
        ax.set_title(title)
    if ylim is not None:
        ax.set_ylim(*ylim)
    if xlim is not None:
        ax.set_xlim(*xlim)
    ax.margins(x=0.01)
    ax.legend(loc=legend_loc)
    if save:
        save_fig(fig, save)
    return fig, ax

## 1. Explained variance

How well the value head predicts returns: `1` is perfect, `0` is no better than
predicting the mean, `< 0` is worse. The run is labelled **PPO Decoder Fine
Tuning**.


In [ ]:
fig, ax = plot_runs(
    [{"run": "fixcritic", "label": "PPO Decoder Fine Tuning", "color": PALETTE["blue"]}],
    tag="losses/explained_variance", ylabel="Explained variance",
    ylim=(-0.05, 1.0), hline=0.0, legend_loc="lower right",
    smooth_from=10_000,  # hide EMA warm-up where early noise skews the line
    # save="explained_variance",
)

## 2. Episodic return

Mean undiscounted return per episode (`charts/episodic_return`), shown two ways:

1. the `fixcritic` run on its own, then
2. both runs overlaid &mdash; **without** vs **with** critic warm-start.

The warm-start run is shorter, so its curve simply ends where its logs stop.


In [ ]:
fig, ax = plot_runs(
    [{"run": "fixcritic", "label": "Without critic warm-start", "color": PALETTE["blue"]}],
    tag="charts/episodic_return", ylabel="Episodic return",
    legend_loc="lower right",
    # save="episodic_return_fixcritic",
)

In [ ]:
fig, ax = plot_runs(
    [{"run": "fixcritic", "label": "Without critic warm-start", "color": PALETTE["blue"]},
     {"run": "warmstart", "label": "With critic warm-start", "color": PALETTE["red"]}],
    tag="charts/episodic_return", ylabel="Episodic return",
    smooth_from=20_000,  # hide EMA warm-up where early noise skews the line
    legend_loc="lower right",
    # save="episodic_return_comparison",
)

# Finished route / success rate

In [ ]:
fig, ax = plot_runs(
    [{"run": "TFV6_LOCAL_RESIDUAL_onlyspeed_newarchitecture_CNN", "label": "PPO Residual Fine Tuning Only Speed", "color": PALETTE["blue"]}],
    tag="infractions/finished_route", ylabel="Average finished route ratio (0-1)",
    ylim=(-0.02, 1.0), band_clip=(0, 1),  # a rate: keep band inside [0, 1]
    smooth_from=20_000,  # hide EMA warm-up where early noise skews the line
    legend_loc="upper right",
    # save="finished_route",
)

In [ ]:
# Baseline and Tfv6 frozen
# fig, ax = plot_runs(
#     [{"run": "TD3_idun_baseline", "label": "TD3 Residual Baseline", "color": PALETTE["blue"]},
#      {"run": "TD3_idun_basepolicy_noexplore", "label": "Frozen TFv6", "color": PALETTE["green"]}],
#     tag="infractions/finished_route", ylabel="Average finished route ratio (0-1)",
#     xlim=(0, 1.51),
#     ylim=(-0.02, 1.0), band_clip=(0, 1),  # a rate: keep band inside [0, 1]
#     smooth_from=20_000,  # hide EMA warm-up where early noise skews the line
#     legend_loc="upper right",
#     # save="finished_route",
# )
fig, ax = plot_runs(
    [{"run": "TD3_idun_baseline", "label": "TD3 Residual Baseline", "color": PALETTE["blue"]},
     {"run": "TD3_idun_baseline_nstep3", "label": "+3-step returns", "color": PALETTE["red"]}],
    tag="infractions/finished_route", ylabel="Average finished route ratio (0-1)",
    figsize=(5.9*0.75, 3.6*0.75),
    xlim=(0, 1.51),
    ylim=(-0.02, 1.0), band_clip=(0, 1),  # a rate: keep band inside [0, 1]
    smooth_from=20_000,  # hide EMA warm-up where early noise skews the line
    legend_loc="lower right",
    # save="finished_route",
)

In [ ]:
# Baseline and Tfv6 frozen
fig, ax = plot_runs(
    [{"run": "TD3_idun_baseline", "label": "TD3 Residual Baseline", "color": PALETTE["blue"]},
     {"run": "TD3_idun_basepolicy_noexplore", "label": "Frozen TFv6", "color": PALETTE["green"]}],
    tag="infractions/collision", ylabel="Average collision rate (0-1)",
    figsize=(5.9/0.75/3, 3.6),
    xlim=(0, 1.51),
    ylim=(-0.02, 1.0), band_clip=(0, 1),  # a rate: keep band inside [0, 1]
    smooth_from=20_000,  # hide EMA warm-up where early noise skews the line
    legend_loc="upper right",
    title="Collision rates",
    # save="finished_route",
)
fig, ax = plot_runs(
    [{"run": "TD3_idun_baseline", "label": "TD3 Residual Baseline", "color": PALETTE["blue"]},
     {"run": "TD3_idun_basepolicy_noexplore", "label": "Frozen TFv6", "color": PALETTE["green"]}],
    tag="infractions/ran_red_light", ylabel="Average red light violation rate (0-1)",
    figsize=(5.9/0.75/3, 3.6),
    xlim=(0, 1.51),
    ylim=(-0.02, 1.0), band_clip=(0, 1),  # a rate: keep band inside [0, 1]
    smooth_from=20_000,  # hide EMA warm-up where early noise skews the line
    legend_loc="upper right",
    title="Red light violation rates",
    # save="finished_route",
)
fig, ax = plot_runs(
    [{"run": "TD3_idun_baseline", "label": "TD3 Residual Baseline", "color": PALETTE["blue"]},
     {"run": "TD3_idun_basepolicy_noexplore", "label": "Frozen TFv6", "color": PALETTE["green"]}],
    tag="infractions/ran_stop_sign", ylabel="Average stop sign violation rate (0-1)",
    figsize=(5.9/0.75/3, 3.6),
    xlim=(0, 1.51),
    ylim=(-0.02, 1.0), band_clip=(0, 1),  # a rate: keep band inside [0, 1]
    smooth_from=20_000,  # hide EMA warm-up where early noise skews the line
    legend_loc="upper right",
    title="Stop sign violation rates",
    # save="finished_route",
)

In [ ]:
# On-demand graphs
fig, ax = plot_runs(
    [{"run": "TD3_idun_baseline", "label": "TD3 Residual Baseline", "color": PALETTE["blue"]},
     {"run": "TD3_idun_baseline_speedhistory", "label": "+Speed History", "color": PALETTE["red"]}],
    tag="infractions/ran_stop_sign", ylabel="Average stop sign violation rate (0-1)",
    figsize=(5.9*0.75, 3.6*0.75),
    xlim=(0, 1.51),
    ylim=(-0.02, 1.0), band_clip=(0, 1),  # a rate: keep band inside [0, 1]
    smooth_from=20_000,  # hide EMA warm-up where early noise skews the line
    legend_loc="upper right",
    title="Stop sign violation rates",
    # save="finished_route",
)

In [ ]:
fig, ax = plot_runs(
    [{"run": "TD3_idun_baseline", "label": "TD3 Residual Baseline", "color": PALETTE["blue"]}],
    tag="residual/speed_coeff_mean", ylabel="Mean predicted speed coefficient (-1, 1)",
    figsize=(5.9/0.75/2, 3.6),
    xlim=(0, 1.51),
    ylim=(-0.02, 1.0), band_clip=(0, 1),  # a rate: keep band inside [0, 1]
    smooth_from=20_000,  # hide EMA warm-up where early noise skews the line
    legend_loc="upper right",
    title="Mean speed coefficients",
    # save="finished_route",
)
fig, ax = plot_runs(
    [{"run": "TD3_idun_baseline", "label": "TD3 Residual Baseline", "color": PALETTE["blue"]}],
    tag="residual/speed_coeff_std", ylabel="Standard deviation of the predicted speed coef.",
    figsize=(5.9/0.75/2, 3.6),
    xlim=(0, 1.51),
    ylim=(-0.02, 1.0), band_clip=(0, 1),  # a rate: keep band inside [0, 1]
    smooth_from=20_000,  # hide EMA warm-up where early noise skews the line
    legend_loc="upper right",
    title="Standard deviation of speed coefficients",
    # save="finished_route",
)

# Effective collision rate

Raw collision rate is misleading when stop signs / red lights are frequent: an
agent that brakes early and never enters the intersection logs few collisions
simply because it rarely gets the chance. The **effective collision rate**
conditions on the episodes where the ego actually negotiated the intersection:

$$\text{eff. collision} = \frac{P(\text{collision})}{P(\text{entered})}
  = \frac{\text{collision}}{\text{collision} + \text{finished\_route}}.$$


To keep it clean we smooth as a **ratio of EMAs** (smoothed collisions over
smoothed entries) rather than EMA-ing the per-update ratio &mdash; the latter is
very noisy when an update contains only a handful of entered episodes.

In [ ]:
# Raw vs effective collision rate for the TD3 residual baseline.
fig, ax = plot_effective_collision(
    [{"run": "TD3_idun_baseline", "label": "TD3 Residual Baseline", "color": PALETTE["blue"]},
     {"run": "TD3_idun_baseline_nstep3", "label": "+3-step returns", "color": PALETTE["green"]}],
    xlim=(0, 1.51),
    figsize=(5.9*0.75, 3.6*0.75),
    smooth_from=20_000,  # hide EMA warm-up where early noise skews the line
    title="Effective collision rate (per entered intersection)",
    # save="effective_collision",
)
# fig, ax = plot_effective_collision(
#     [{"run": "TD3_idun_baseline", "label": "TD3 Residual Baseline", "color": PALETTE["blue"]},
#      {"run": "TD3_idun_basepolicy_noexplore", "label": "Frozen TFv6", "color": PALETTE["green"]}],
#     xlim=(0, 1.51),
#     figsize=(5.9*0.75, 3.6*0.75),
#     smooth_from=20_000,  # hide EMA warm-up where early noise skews the line
#     title="Effective collision rate (per entered intersection)",
#     # save="effective_collision",
# )

## Reuse &amp; extending

Add a run, then plot it:

```python
RUNS["myrun"] = "TFV6_PPO_LOCAL_only_NSLTEF_something"
plot_runs(
    [{"run": "myrun", "label": "My run"}],
    tag="charts/episodic_return", ylabel="Episodic return", save="myrun_return",
)
```

Handy knobs on `plot_runs` / `add_trace`:

- `smoothing` &ndash; EMA weight in 0&ndash;1 (higher = smoother), used for both
  the bold line and the band centre.
- `band` &ndash; `"deviation"` (default: centre &plusmn; EMA-smoothed local std,
  always smooth, best for sparse / seldom-reported signals), or the window-based
  `"percentile"` / `"std"` / `"minmax"` (edges are EMA-smoothed via
  `band_smooth`), or `None` to hide the band.
- `band_k` &ndash; width of the deviation band in std units (default `1.0`).
- `band_clip` &ndash; clamp the band to a valid range, e.g. `(0, 1)` for a rate.
- `smooth_from` &ndash; hide the bold line before this raw step (e.g. `10_000`)
  to drop the EMA warm-up.
- `band_kw` &ndash; window options for the window-based kinds, e.g.
  `{"window": 61, "q": (10, 90)}`.
- plus `show_raw`, `xdiv`, `ylim`, `hline`.

Call `list_tags("warmstart")` to see every available scalar tag.